# kNN recommendation and the student-project handoff

**Lecture 13 · Notebook 02 · CMOR 438 / INDE 577**

This laboratory moves from prediction to retrieval. We use authentic anonymized MovieLens ratings to build item-item collaborative filtering, evaluate recommendation evidence, and then map the work into the companion student project template.

## How to use this notebook

Run the core route to inspect events, fit item similarities, and produce recommendations. Then follow the project-handoff checklist in your own template repository. Do not copy output cells as evidence. Commit retrieval code, provenance, package code, tests, and a self-contained applied notebook through a reviewed pull request.

## Learning objectives

You will distinguish content-based and collaborative recommendation, construct a user-item matrix, derive adjusted-cosine item similarity, prevent already-seen recommendations, evaluate ranking beyond RMSE, and place data, estimator, tests, configuration, and narrative in professional repository boundaries.

## Why this matters

Recommendation systems retrieve a small set of useful items from a large catalog. Accuracy alone is inadequate: relevance, novelty, diversity, coverage, latency, popularity bias, feedback loops, and user agency all matter. kNN makes the retrieval mechanism inspectable before modern embedding and deep-ranking systems add complexity.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from rice_dsm.paths import find_project_root

PROJECT_ROOT = find_project_root()
EXPECTED_ENVIRONMENT = PROJECT_ROOT / ".venv"

# A notebook kernel is a long-running Python process.  Its executable—not the
# terminal prompt—determines which packages this notebook imports.
print("Python version:    ", sys.version.split()[0])
print("Python executable: ", sys.executable)
print("Active prefix:     ", sys.prefix)
print("Repository root:   ", PROJECT_ROOT)
print("Using project .venv:", Path(sys.prefix).resolve() == EXPECTED_ENVIRONMENT.resolve())

if Path(sys.prefix).resolve() != EXPECTED_ENVIRONMENT.resolve():
    raise RuntimeError(
        "This notebook is not using the repository .venv. In VS Code choose "
        "Kernel > Select Another Kernel > Jupyter Kernel > Rice DSM. If it is "
        "missing, close the notebook, run `uv sync --locked` and "
        "`uv run python scripts/setup_course.py` in the repository terminal, "
        "then reopen the notebook."
    )

sns.set_theme(style="whitegrid", context="notebook")

## 1. Data contract: MovieLens rating events

The committed teaching subset is derived deterministically from GroupLens MovieLens `ml-latest-small`: all rating events for the 80 most-rated movies. One row in `ratings.csv` is an anonymized user assigning a 0.5–5.0 star rating to one movie at a Unix timestamp. One row in `movies.csv` is a movie identifier, manually entered title, and pipe-separated genre list.

The full source contains 100,836 ratings from 610 users across 9,742 movies. The bounded subset is for executable teaching, not benchmark publication. It amplifies popularity bias by construction and must not support claims about all users, movies, or cultures. Read `data/README.md` and the upstream usage license before reuse.

In [ ]:
from rice_dsm.paths import movielens_teaching_directory

data_directory = movielens_teaching_directory()
ratings_path = data_directory / "ratings.csv"
movies_path = data_directory / "movies.csv"

# Loading is explicit: the notebook does not hide a network request or mutate
# the source files.  Students will use data/local in their template repository.
ratings = pd.read_csv(ratings_path)
movies = pd.read_csv(movies_path)

print("rating events:", len(ratings))
print("users:", ratings["userId"].nunique())
print("movies:", ratings["movieId"].nunique())
display(ratings.head())
display(movies.head())

assert len(ratings) == 13_817
assert ratings[["userId", "movieId"]].duplicated().sum() == 0
assert set(ratings["movieId"]) == set(movies["movieId"])

## 2. Explore with a purpose

We ask four questions before fitting:

1. Is rating volume concentrated among users or items?
2. Is the 0.5–5.0 response distribution balanced?
3. How sparse is the user-item matrix?
4. What selection mechanism created this teaching subset?

These checks anticipate cold start, popularity bias, weak overlap, and misleading offline metrics.

In [ ]:
user_activity = ratings.groupby("userId").size().sort_values(ascending=False)
movie_popularity = ratings.groupby("movieId").size().sort_values(ascending=False)
density = len(ratings) / (ratings["userId"].nunique() * ratings["movieId"].nunique())

print(f"user-item density: {density:.3%}")
print("least / median / most ratings per user:",
      int(user_activity.min()), int(user_activity.median()), int(user_activity.max()))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
sns.histplot(ratings, x="rating", discrete=True, ax=axes[0], color="#00205B")
axes[0].set(title="Explicit rating distribution", xlabel="Stars")
sns.ecdfplot(x=user_activity, ax=axes[1], color="#C8102E")
axes[1].set(title="User activity is unequal", xlabel="Ratings per user")
sns.ecdfplot(x=movie_popularity, ax=axes[2], color="#F2A900")
axes[2].set(title="Movie exposure is unequal", xlabel="Ratings per movie")
plt.show()

## 3. Item-item collaborative filtering: missing does not mean negative

Content-based retrieval compares item attributes such as genre. **Collaborative filtering** compares behavior: two movies are similar when the same users rate them similarly.

Let $r_{ui}$ be user $u$'s rating of item $i$, and $\bar r_u$ the mean of that user's observed ratings. For users $U_{ij}$ who rated both items, adjusted cosine similarity is

$$
s(i,j)=\frac{\sum_{u\in U_{ij}}(r_{ui}-\bar r_u)(r_{uj}-\bar r_u)}
{\sqrt{\sum_{u\in U_{ij}}(r_{ui}-\bar r_u)^2}
 \sqrt{\sum_{u\in U_{ij}}(r_{uj}-\bar r_u)^2}}.
$$

Centering by user removes some generous-versus-strict rating behavior. We require a minimum overlap because a high similarity from one co-rating is fragile.

In [ ]:
from rice_dsm.ml.neighbors import ItemKNNRecommender

# The estimator receives event columns and owns the fitted matrix and similarity state.
recommender = ItemKNNRecommender(n_neighbors=15, min_common_users=8).fit(
    ratings["userId"].to_numpy(),
    ratings["movieId"].to_numpy(),
    ratings["rating"].to_numpy(),
)

print("user-item matrix shape:", recommender.ratings_.shape)
print("item-similarity shape:", recommender.similarities_.shape)
assert recommender.ratings_.shape == (ratings.userId.nunique(), 80)

In [ ]:
title_by_id = movies.set_index("movieId")["title"]

def show_similar(title_fragment: str, n: int = 8) -> pd.DataFrame:
    """Display positively similar movies for one unambiguous title fragment."""
    matches = movies[movies["title"].str.contains(title_fragment, case=False, regex=False)]
    if len(matches) != 1:
        raise ValueError(f"Expected one title containing {title_fragment!r}; found {len(matches)}")
    movie_id = int(matches.iloc[0]["movieId"])
    neighbor_ids, similarities = recommender.similar_items(movie_id, n=n)
    return pd.DataFrame({
        "movie": title_by_id.loc[neighbor_ids].to_numpy(),
        "adjusted_cosine_similarity": similarities,
    })

display(show_similar("Toy Story (1995)"))

### Worked example: personalized scoring

For user $u$ and unseen candidate $j$, take up to $k$ positively similar movies that $u$ rated. A simple score is

$$
\widehat r_{uj}=\bar r_u+
\frac{\sum_{i\in N_k(j;u)}s(j,i)(r_{ui}-\bar r_u)}
{\sum_{i\in N_k(j;u)}|s(j,i)|}.
$$

The candidate is not recommended when it has no positive supported neighbor. Already-rated items are excluded. This is a transparent teaching policy, not a complete production ranking system.

In [ ]:
def recommendations_for(user_id: int, n: int = 10) -> pd.DataFrame:
    item_ids, predicted_ratings = recommender.recommend(user_id, n=n)
    return pd.DataFrame({
        "movieId": item_ids,
        "title": title_by_id.loc[item_ids].to_numpy(),
        "predicted_rating": predicted_ratings,
    })

query_user = 1
user_history = ratings[ratings["userId"] == query_user].merge(movies, on="movieId")
display(user_history.sort_values("rating", ascending=False)[["title", "rating"]].head(10))
display(recommendations_for(query_user))

recommended_ids = set(recommendations_for(query_user)["movieId"])
seen_ids = set(user_history["movieId"])
assert recommended_ids.isdisjoint(seen_ids)

## 4. Offline evaluation is a counterfactual approximation

We cannot observe how a user would have rated every unseen movie. A temporal holdout can hide each eligible user's latest positive event, train only on earlier events, and ask whether the hidden item appears in the top-$N$ list. `Recall@N` or hit rate then measures retrieval of known positives, not satisfaction. Exposure is missing-not-at-random: an unrated movie is not necessarily disliked.

Also report catalog coverage and popularity concentration. A recommender that suggests the same blockbuster to everyone can achieve a plausible hit rate while providing little discovery.

In [ ]:
# Build one temporal positive holdout for users with enough history.
ordered = ratings.sort_values(["userId", "timestamp"])
eligible_counts = ordered.groupby("userId").size()
eligible_users = eligible_counts[eligible_counts >= 12].index[:60]
eligible_events = ordered[ordered["userId"].isin(eligible_users)]
holdout_indices = eligible_events.groupby("userId").tail(1).index

training_events = ratings.drop(index=holdout_indices)
holdout = ratings.loc[holdout_indices, ["userId", "movieId", "rating"]]
evaluation_model = ItemKNNRecommender(n_neighbors=15, min_common_users=8).fit(
    training_events["userId"], training_events["movieId"], training_events["rating"]
)

hits = []
recommended_catalog = set()
for event in holdout.itertuples(index=False):
    recommended_ids, _ = evaluation_model.recommend(event.userId, n=10)
    recommended_set = set(recommended_ids.tolist())
    recommended_catalog.update(recommended_set)
    hits.append(event.movieId in recommended_set)

evaluation = pd.Series({
    "users_evaluated": len(hits),
    "recall_at_10": np.mean(hits),
    "catalog_coverage_at_10": len(recommended_catalog) / movies.movieId.nunique(),
})
evaluation

## 5. Professional practice: data and artifact boundaries

Do not serialize raw user histories into a model artifact without an explicit privacy and retention design. The fitted recommender contains interaction data. Access control, deletion, audit, and retraining requirements follow it.

For a production-scale catalog, the $m\times m$ similarity matrix costs $O(m^2)$ storage. Sparse methods, approximate indexes, embeddings, candidate generation, and online ranking become separate components. Monitor recommendation latency, empty-result rate, coverage, popularity concentration, drift, and user feedback—not only offline accuracy.

## 6. The student-template implementation map

Create your project from the [companion template](https://github.com/RandyRDavila/Data-Science-Machine-Learning-Project-Template), then work on a focused branch. The target layout is:

```text
data/local/your_dataset/                    # ignored full/local data
data/fixtures/neighbors/                    # tiny lawful CI fixture
docs/DATASET_CARD.md                        # source, license, row meaning, limits
configs/student.toml                        # ignored local paths and task config
scripts/fetch_or_validate_your_data.py      # reproducible retrieval/validation
src/dsm_project/algorithms/student/neighbors.py
src/dsm_project/algorithms/registry.py      # explicit registration
tests/test_student_neighbors.py
notebooks/NN-your-question-not-knn.ipynb    # applied scientific/product story
```

The notebook should be named for the question, not merely the algorithm. Reusable distance, fit, prediction, recommendation, and metric logic belongs under `src/`; the notebook imports that package and explains the investigation.

### Exact project workflow

From a clean template `main`:

```text
git switch main
git pull --ff-only
git switch -c YOUR-GITHUB-USERNAME/knn-case-study
uv sync --locked
```

Then:

1. write the question and data contract in an issue;
2. place the full dataset under ignored `data/local/` or retrieve it by script;
3. commit only a small lawful fixture needed by CI;
4. complete `docs/DATASET_CARD.md` before modeling;
5. implement typed, NumPy-style documented code in the student package;
6. register the estimator explicitly;
7. write analytic and failure tests before the applied notebook;
8. import the package from the notebook—never duplicate the class there;
9. run `uv run ruff check src tests scripts` and `uv run pytest -q`;
10. commit, push, and open a focused PR that links the issue and interprets evidence.

“Upload the data” does not mean commit any file you found. License, privacy, size, provenance, and reproducibility determine whether you commit a fixture, retrieval code, a database snapshot, or only metadata.

### Test plan for a student kNN contribution

| Test | Evidence |
| --- | --- |
| $3$-$4$-$5$ distance | analytic unit correctness |
| hand vote and hand mean | classifier/regressor aggregation |
| sorted neighbor distances | structural property |
| exact duplicate under distance weighting | zero-distance policy |
| invalid $k$, shape, NaN, pre-fit predict | user-facing failures |
| agreed scikit-learn configuration | differential behavior |
| already-seen items excluded | recommendation invariant |
| tiny fixture through configured experiment | package integration |
| clean-kernel notebook execution | environment and narrative reproducibility |

Tests should protect behavior, not private helper names. A failing test should tell a teammate which contract broke.

In [ ]:
# This cell demonstrates the import boundary students should reproduce.
# In the template repository, the import becomes:
# from dsm_project.algorithms.student.neighbors import KNearestNeighborsClassifier
from rice_dsm.ml.neighbors import KNearestNeighborsClassifier as CourseKNN

smoke_model = CourseKNN(n_neighbors=1).fit([[0.0], [1.0]], ["low", "high"])
assert smoke_model.predict([[0.1]]).tolist() == ["low"]

print("The notebook imported package code; it did not redefine the estimator.")

## Debugging clinic and common failure modes

| Symptom | Likely cause | Evidence to inspect |
| --- | --- | --- |
| Empty recommendations | insufficient overlap or positive similarity | co-rating counts and similarity distribution |
| Same list for everyone | popularity dominates or personalization is absent | per-user list diversity |
| Great offline recall, poor user response | exposure bias or proxy mismatch | online outcomes and user feedback |
| CI cannot find data | notebook reads `Downloads/` or an absolute path | fixture and config boundary |
| Import works only in one notebook | implementation was copied into a cell | package path and clean-kernel test |
| PR leaks a large/private file | data policy was considered after `git add` | `git status`, `.gitignore`, dataset card |

**Common failure:** recommendation is not classification with movie titles as labels. The system ranks unseen candidates under incomplete, behaviorally selected feedback.

## Guided practice

1. Inspect recommendations for three users with different histories. What repeats?
2. Change `min_common_users` and graph recall versus coverage.
3. Add a popularity baseline that recommends the most-rated unseen movies. Does kNN beat it?
4. Explain why random event splitting can place a future interaction in training for an earlier recommendation.

In [ ]:
# PRACTICE: construct the popularity baseline for the same holdout users.
popular_items = training_events.groupby("movieId").size().sort_values(ascending=False).index
popularity_hits = []
for event in holdout.itertuples(index=False):
    seen = set(training_events.loc[training_events.userId == event.userId, "movieId"])
    top_unseen = [item for item in popular_items if item not in seen][:10]
    popularity_hits.append(event.movieId in top_unseen)

print("kNN recall@10:       ", np.mean(hits))
print("popularity recall@10:", np.mean(popularity_hits))
assert len(popularity_hits) == len(hits)

## Independent practice

In your project repository, open an issue proposing a kNN use that serves your actual question. Specify the observation and prediction units, distance, scaling, $k$ selection, baseline, split rule, metric, latency envelope, and one subgroup or coverage risk. If kNN is scientifically inappropriate, write the incompatibility argument instead of forcing it into the product.

## Extension: two-stage recommendation systems

Modern recommenders often separate **candidate generation** from **ranking**. An approximate neighbor index can retrieve hundreds of plausible candidates; a learned ranker then uses context, freshness, constraints, and richer features. Draw the two-stage interface, state which artifact versions must be logged, and design a fallback when retrieval returns no supported items.

## Retrieval practice

1. Why user-center ratings before item similarity?
2. Why require multiple co-raters?
3. Why are unrated items not negative labels?
4. Which files belong under `data/local`, `data/fixtures`, `src`, `tests`, and `notebooks`?
5. What must a PR demonstrate before a student estimator is promoted into product behavior?

## Takeaway

kNN recommendation is retrieval from behavioral similarity, not merely another label prediction. Its fitted state can contain sensitive histories; its offline evidence is biased by exposure; and its value depends on coverage, diversity, latency, and user outcomes. The professional workflow keeps data provenance, reusable algorithms, tests, configuration, notebooks, and review evidence in explicit boundaries.

## Further reading

- [GroupLens MovieLens datasets and usage terms](https://grouplens.org/datasets/movielens/)
- Harper and Konstan, “The MovieLens Datasets: History and Context” (2015).
- [scikit-learn nearest-neighbor guide](https://scikit-learn.org/stable/modules/neighbors.html)
- [Python packaging tutorial](https://packaging.python.org/en/latest/tutorials/packaging-projects/)
- [Python `pathlib` documentation](https://docs.python.org/3/library/pathlib.html)